In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.model_selection import GroupKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge

ROOT = Path.cwd().parents[1]

df = pd.read_csv(ROOT / "data" / "duolingo_flagship_v5.csv")
split_df = pd.read_csv(ROOT / "data" / "split_users.csv")

cv_users = set(split_df.loc[split_df["split"] == "cv", "user_id"])

df_cv = df[df["user_id"].isin(cv_users)].copy()

features = ["lag_days","history_seen","history_correct","history_accuracy","lag_days_log"]

X = df_cv[features]
y = df_cv["p_recall"]
groups = df_cv["user_id"]

print(X.shape, groups.nunique())

(14438, 5) 2125


In [3]:
pipeline = Pipeline([
    ("scaler",StandardScaler()),
    ("model",Ridge())
])

param_grid = {"model__alpha" : [ 0.01,0.1,1,3,10,30,100,300]}

gkf = GroupKFold(n_splits=5)

grid = GridSearchCV(estimator=pipeline,param_grid=param_grid,scoring="neg_root_mean_squared_error",cv=gkf,n_jobs=-1)
grid.fit(X,y,groups=groups)

print("Best alpha:", grid.best_params_["model__alpha"])
print("Best CV RMSE:", -grid.best_score_)

Best alpha: 30
Best CV RMSE: 0.27346817695193787


In [4]:
from sklearn.linear_model import Lasso

lasso_pipeline = Pipeline([("scaler", StandardScaler()),("model", Lasso(max_iter=10000))])
lasso_param_grid = {"model__alpha": [0.00001,0.00003,0.0001,0.0003,0.001,0.003,0.01,0.03, 0.1]}
lasso_grid = GridSearchCV(estimator=lasso_pipeline,param_grid=lasso_param_grid,scoring="neg_root_mean_squared_error",cv=gkf, n_jobs=-1)
lasso_grid.fit(X,y,groups=groups)

print("Best alpha:", lasso_grid.best_params_["model__alpha"])
print("Best CV RMSE:", -lasso_grid.best_score_)

Best alpha: 0.0001
Best CV RMSE: 0.2735331529309209


In [5]:
ridge_idx = grid.best_index_
lasso_idx = lasso_grid.best_index_

ridge_mean = -grid.cv_results_["mean_test_score"][ridge_idx]
ridge_std = grid.cv_results_["std_test_score"][ridge_idx]

lasso_mean = -lasso_grid.cv_results_["mean_test_score"][lasso_idx]
lasso_std = lasso_grid.cv_results_["std_test_score"][lasso_idx]

print(f"Ridge: {ridge_mean:.6f} ± {ridge_std:.6f}")
print(f"Lasso: {lasso_mean:.6f} ± {lasso_std:.6f}")

Ridge: 0.273468 ± 0.010290
Lasso: 0.273533 ± 0.010116


## Part 12 Conclusion

Hyperparameters were selected using 5-fold `GroupKFold` on the CV pool.

The final tuning results were:

| Model | Best alpha | CV RMSE |
|---|---:|---:|
| Linear Regression | — | `0.273699 ± 0.009834` |
| Ridge | `30` | `0.273468 ± 0.010290` |
| Lasso | `0.0001` | `0.273533 ± 0.010116` |

Ridge achieved the lowest mean CV RMSE among the tested models.

However, the improvement over ordinary linear regression is very small relative to the fold-to-fold variability, so it should not be interpreted as a large performance gain.

Lasso selected a very small alpha, suggesting that strong sparsity is not beneficial for the current five-feature representation.

The holdout test set was not used for hyperparameter selection and remains untouched.

The main result is that regularization provides only a small improvement within the current linear model family. Combined with the learning-curve diagnosis from Part 11, this suggests that future gains are more likely to come from richer features and nonlinear models than from further tuning of linear models.